# 💓 Path 1: Signal-to-Class (1D ECG AI Classification)

**Research Context**: This path explores the most direct form of AI diagnosis—processing raw electrical telemetry. We use 1D Convolutional Neural Networks to mimic a cardiologist's pattern recognition.

### 🌟 Supervisor's Guide:
1.  **Diagnostic Focus**: We overlay the AI's 'attention' (heatmap) on the raw signal to see which peaks determine the classification.
2.  **Clinical Relevance**: Using real MIT-BIH data to distinguish between Normal, Ventricular, and Supraventricular beats.
3.  **Model Transparency**: Seeing the internal 'filter response' that leads to a diagnosis.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

# Set seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Using device: {device}")

## 📊 1. Loading Real MIT-BIH Data
We load a balanced slice of the dataset (187 points per heartbeat).

In [ ]:
# Flexible Path Handling for local or Colab/Kaggle environments
DATA_PATH = "sample_mitbih.csv"
possible_paths = [
    "sample_mitbih.csv",
    "/content/sample_mitbih.csv",
    "notebooks/supervisor/sample_mitbih.csv",
    "../supervisor/sample_mitbih.csv"
]

for path in possible_paths:
    if os.path.exists(path):
        DATA_PATH = path
        break

if os.path.exists(DATA_PATH):
    print(f"📂 Loading real data from: {DATA_PATH}")
    df = pd.read_csv(DATA_PATH, header=None)
else:
    print("⚠️ Warning: sample_mitbih.csv not found. Using synthetic data for structure demo.")
    df = pd.DataFrame(np.random.randn(100, 188))
    # Ensure we have all 5 classes in dummy data
    df.iloc[:, 187] = np.tile(np.arange(5), 20)

X_raw = df.iloc[:, :187].values.astype(np.float32)
y_raw = df.iloc[:, 187].values.astype(np.int64)

class_names = ["Normal", "SVEB", "VEB", "Fusion", "Unknown/Paced"]
print(f"✅ Loaded {len(X_raw)} heartbeats.")

# Show samples - Robust plotting logic
plt.figure(figsize=(15, 6), facecolor='#f8f9fa')
for i in range(5):
    indices = np.where(y_raw == i)[0]
    plt.subplot(2, 3, i+1)
    if len(indices) > 0:
        idx = indices[0]
        plt.plot(X_raw[idx], color='#007bff')
        plt.title(f"Class: {class_names[i]}", fontsize=12)
    else:
        plt.text(0.5, 0.5, f"Class {class_names[i]}\nnot found in sample", 
                 ha='center', va='center', color='red')
        plt.title(f"Class: {class_names[i]} (Missing)", fontsize=12)
    plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

## 🧠 2. Simple 1D CNN Architecture
This model uses 1D convolutions to scan the signal for patterns.

In [ ]:
class SimpleCNN1D(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(16, 32, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 46, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )
        
    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

model = SimpleCNN1D().to(device)
print(model)

## 🎓 3. Training on Real Samples
We train for 15 epochs to observe the convergence.

In [ ]:
X_tensor = torch.tensor(X_raw, dtype=torch.float32).unsqueeze(1).to(device)
y_tensor = torch.tensor(y_raw, dtype=torch.long).to(device)
loader = DataLoader(TensorDataset(X_tensor, y_tensor), batch_size=32, shuffle=True)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("🚀 Training...")
for epoch in range(15):
    model.train()
    total_loss = 0
    for signals, labels in loader:
        optimizer.zero_grad()
        outputs = model(signals)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1}/15, Loss: {total_loss/len(loader):.4f}")

print("✅ Training Complete!")

### 🏥 Note on 'Unknown' Class (Category Q):
The **Unknown** class is a standard clinical category in the MIT-BIH dataset. It includes:
- **Paced beats** (from artificial pacemakers)
- **Unclassifiable signals** (distorted by noise or artifacts)

In a real-world deployment, 'Unknown' triggers a **manual review** by a cardiologist, ensuring AI safety.

## 🔍 4. Internal Mechanism: Visualizing Activations
What does the AI see? We plot the output of the convolutional filters.

In [ ]:
def visualize_diagnostic_focus(model, sample_x, layer_idx=0):
    activations = []
    def hook(module, input, output):
        activations.append(output.detach().cpu().numpy())
    
    handle = model.features[layer_idx].register_forward_hook(hook)
    model(sample_x.to(device))
    handle.remove()
    
    # Average activations across filters to see general 'attention'
    act = np.mean(activations[0][0], axis=0)
    act_normalized = (act - act.min()) / (act.max() - act.min() + 1e-8)
    
    signal = sample_x[0, 0].cpu().numpy()
    
    plt.figure(figsize=(15, 6), facecolor='#fdfdfd')
    
    # Plotting the signal with a heatmap overlay
    plt.subplot(2, 1, 1)
    plt.plot(signal, color='black', alpha=0.3, label='ECG Trace')
    x_range = np.linspace(0, len(signal)-1, len(act))
    plt.scatter(x_range, signal[x_range.astype(int)], c=act, cmap='Reds', s=50, alpha=0.8, label='AI Attention')
    plt.title("1. Diagnostic Focus: Where the AI 'Looks' to Classify", fontsize=14, fontweight='bold')
    plt.legend()
    
    plt.subplot(2, 1, 2)
    plt.imshow(activations[0][0], aspect='auto', cmap='magma')
    plt.title(f"2. Neural Filter Responses (Layer {layer_idx})", fontsize=14, fontweight='bold')
    plt.xlabel("Time Steps")
    plt.ylabel("Filter Index")
    plt.colorbar(label="Activation Strength")
    
    plt.tight_layout()
    plt.show()

# Visualize for a class 2 (Ventricular) heartbeat
v_indices = np.where(y_raw == 2)[0]
target_idx = v_indices[0] if len(v_indices) > 0 else 0
print(f"🔍 Visualizing diagnostic focus for Class: {class_names[y_raw[target_idx]]}")
visualize_diagnostic_focus(model, X_tensor[target_idx:target_idx+1], layer_idx=0)